In [70]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
import os
from dotenv import load_dotenv
load_dotenv()


True

In [71]:
embedding = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
splitter= RecursiveCharacterTextSplitter(chunk_size=200)

In [72]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("dataset.txt", encoding="utf-8")
documents = loader.load()

In [73]:
chunks = splitter.split_documents(documents)
print(len(chunks))

14


In [74]:
vector_store= FAISS.from_documents(chunks,embedding)

In [75]:
vector_store.index_to_docstore_id

{0: '58bfa343-7f3b-4b55-bb2e-a31f9ee0b7b7',
 1: '5204040b-3164-4d00-8b97-323580ef4f0f',
 2: 'bca31f83-75b8-4f65-ba3b-3cc6f622a3e0',
 3: 'a00efe08-cb81-4eb9-99ad-844e1c60d959',
 4: '0d6bef73-5245-48fc-a330-872a13b4aa4b',
 5: 'e2d87f55-0b62-4cd0-a80e-412bac677a31',
 6: '59835840-1001-497f-8718-0c2f88d5c937',
 7: '8c1666e7-ab6c-4798-a961-ce21b1b6a4a6',
 8: '50ce2548-7d46-437a-a3e0-b79afbd42223',
 9: 'a35c89c7-61da-4f51-9709-992e14bd73f4',
 10: '1ae21550-dde3-40b7-8303-893ca2c666d9',
 11: '3f765ae4-8bc7-4571-8431-d8d3f3b06ac2',
 12: '07bb08bb-105e-41a6-9553-cddefe65e5ad',
 13: '3165bd8f-5e5e-44e1-b481-fda335809920'}

In [76]:
vector_store.get_by_ids(['0490050e-f3a6-4771-bc2e-6f0032679a93'])

[]

In [77]:
retriever= vector_store.as_retriever(search_type='similarity', search_kwargs={'k':1})
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x1196d2690>, search_kwargs={'k': 1})

In [78]:
retriever.invoke('what is KYC')

[Document(id='58bfa343-7f3b-4b55-bb2e-a31f9ee0b7b7', metadata={'source': 'dataset.txt'}, page_content='question: What is KYC?\nanswer: KYC (Know Your Customer) is a process banks use to verify the identity of customers.')]

In [79]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint


In [80]:
prompt= PromptTemplate(
    template='''

You are a helpful banking assistant.
Answer ONLY from the provided document context.
If the context is insufficient or irrelevant, just say you don't know.
{context}
Question: {question}''',
    input_variables=['context','question']

)

In [110]:
# question= "UPI"
question=input()
retrieved_docs= retriever.invoke(question)
retrieved_docs

[Document(id='0d6bef73-5245-48fc-a330-872a13b4aa4b', metadata={'source': 'dataset.txt'}, page_content='question: What is an ATM?\nanswer: ATM (Automated Teller Machine) allows customers to withdraw cash and perform basic transactions.')]

In [95]:
final_prompt= prompt.invoke({'context':retrieved_docs, 'question':question})

In [96]:
final_prompt

StringPromptValue(text="\n\nYou are a helpful banking assistant.\nAnswer ONLY from the provided document context.\nIf the context is insufficient or irrelevant, just say you don't know.\n[Document(id='a00efe08-cb81-4eb9-99ad-844e1c60d959', metadata={'source': 'dataset.txt'}, page_content='question: What is a current account?\\nanswer: A current account is used by businesses for frequent transactions with no interest earnings.')]\nQuestion: current account")

In [97]:
llm = HuggingFaceEndpoint(
    repo_id= 'mistralai/Mistral-7B-Instruct-v0.2',
    task= 'text-generation',
    huggingfacehub_api_token= os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN"),
    temperature= 0.6
)

llm= ChatHuggingFace(llm=llm)

answer = llm.invoke(final_prompt)

print(answer.content)

A current account is a type of banking account used primarily by businesses for frequent transactions. Current accounts do not typically earn interest.


In [98]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [99]:
def format_docs(retrieved_docs):
    context_text= '\n\n'.join(doc.page_content for doc in retrieved_docs)
    return context_text

In [100]:
parallel_chain= RunnableParallel({
    'context': retriever|RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
#testing
parallel_chain.invoke('what is upi')

{'context': 'question: What is UPI?\nanswer: UPI (Unified Payments Interface) enables instant money transfers between bank accounts.',
 'question': 'what is upi'}

In [102]:
parser= StrOutputParser()

In [111]:
main_chain=parallel_chain|prompt|llm|parser

In [112]:
print(question)

explain atm in 100 words


In [113]:
main_chain.invoke(question)

"An ATM, or Automated Teller Machine, is a financial machine that enables customers to withdraw cash, check account balances, make transfers between accounts, and perform other basic banking functions. The ATM uses a combination of technology, such as magnetic stripe readers and keypads, and telecommunications networks to connect to a customer's bank account and process transactions. The use of ATMs has revolutionized the way people access their money and perform basic banking functions, making it a convenient and essential tool for managing personal finances."